In [17]:
'''
            UŻYCIE AI

Kod został napisany z wykorzystaniem 
generatywnej AI. Autorzy zweryfikowali
wszystko i prezentują własny wkład intelektualny,
biorąc pełną odpowiedzialność za ostateczną postać pracy
'''

'\n            UŻYCIE AI\n\nKod został napisany z wykorzystaniem \ngeneratywnej AI. Autorzy zweryfikowali\nwszystko i prezentują własny wkład intelektualny,\nbiorąc pełną odpowiedzialność za ostateczną postać pracy\n'

In [53]:
import numpy as np
import pandas as pd

In [54]:
# wczytanie lookupu stref
zones = pd.read_csv("taxi_zone_lookup.csv")

# zostawiamy tylko strefy z Manhattanu
manhattan_zones = zones.loc[zones["Borough"] == "Manhattan", "LocationID"].tolist()

manhattan_zones;

In [55]:
''' 
    WCZYTANIE TAXI. FITLROWANIE. 
    
    Wyrzucamy kolumny "VendorID" i "store_and_fwd_flag".
    Zostawiamy tylko te przejazdy, które:
    - zaczynają i kończą się na Manhattanie. 
    - mają dodatnie / nieujemne wartości w 
        polach gdzie inne nie mają sensu (passenger_count,
        trip_distance, fare_amount, extra, tip_amount, 
        tolls_amount, total_amount)
    - mają congestion_surcharge zgodne z
        https://www.nyc.gov/site/tlc/about/congestion-surcharge.page
        (0 lub 2.5)
    - mają cbd_congestion_fee zgodne z
        https://congestionreliefzone.mta.info/tolling
        (0 lub 0.75)
    - mają RatecodeID standard (1)
        https://www.nyc.gov/site/tlc/passengers/taxi-fare.page
        https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf
    - mają payment_type karta lub gotówka (1 lub 2)
        https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf
    - mają mta_tax równy 0.5 
        https://www.nyc.gov/site/tlc/passengers/taxi-fare.page
    - mają improvement_surcharge równy 1
        https://www.nyc.gov/site/tlc/passengers/taxi-fare.page
    - mają zerowe wartości airport_fee
'''

taxi = (pd.read_parquet("yellow_tripdata_2025-03.parquet")
      .drop(columns = ["VendorID", "store_and_fwd_flag"])
      .query('PULocationID in @manhattan_zones and '
             'DOLocationID in @manhattan_zones and '
             'passenger_count > 0 and '
             'trip_distance > 0 and '
             'fare_amount > 0 and '
             'extra >= 0 and '
             'tip_amount >= 0 and '
             'tolls_amount >= 0 and ' 
             'total_amount >= 0 and '
             'congestion_surcharge in (0, 2.5) and '
             'cbd_congestion_fee in (0, 0.75) and '
             'RatecodeID == 1 and '
             'mta_tax == 0.5 and '
             'improvement_surcharge == 1 and '
             'Airport_fee == 0') 
      )

In [57]:
# Sprawdzamy czy są jakieś brakujące wartości
taxi.isna().values.any()

np.False_

In [58]:
taxi["trip_time_mins"] = (
    taxi["tpep_dropoff_datetime"] - taxi["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

In [59]:
# Usuwamy bezsensowne (niedodatnie) wartości trip_time_mins
taxi = taxi[taxi["trip_time_mins"] > 0]

In [60]:
# Wyciągamy z tpep_pickup_datetime godzinę i dzień tygodnia
#     przypisujemy godzinę na zasadnie + / - pół godziny
taxi = taxi.assign(hour = taxi["tpep_pickup_datetime"].dt.hour,
                   weekday = taxi["tpep_pickup_datetime"].dt.weekday,
                   month = taxi["tpep_pickup_datetime"].dt.month)
# weekday 0 to poniedziałek

In [61]:
# Sprawdzamy czy miesiąc się zgadza
taxi['month'].value_counts()

month
3     2572752
2          25
12          1
4           1
Name: count, dtype: int64

In [62]:
# Usuwamy rekordy z błędnymi miesiącami
taxi = taxi[taxi["month"] == 3]

In [63]:
taxi['month'].value_counts()

month
3    2572752
Name: count, dtype: int64

In [64]:
# Usuwamy kolumny niepotrzebne do modelu
taxi = taxi.drop(columns = ["tpep_dropoff_datetime", "passenger_count",
                          "RatecodeID", "payment_type", "extra",
                          "mta_tax", "tip_amount", "tolls_amount",
                          "improvement_surcharge", "congestion_surcharge",
                          "Airport_fee", "cbd_congestion_fee",
                          "PULocationID", "DOLocationID"])

# Dodajemy kolumnę type, żeby można było połączyć z Uberem
taxi["type"] = "Taxi"

In [65]:
taxi

,tpep_pickup_datetime,trip_distance,fare_amount,total_amount,trip_time_mins,hour,weekday,month,type
0,2025-03-01 00:17:16,0.90,7.9,15.50,8.600000,0,5,3,Taxi
1,2025-03-01 00:37:38,0.60,6.5,13.80,6.216667,0,5,3,Taxi
2,2025-03-01 00:24:35,1.94,14.9,25.81,15.233333,0,5,3,Taxi
3,2025-03-01 00:56:16,0.95,7.2,15.54,5.316667,0,5,3,Taxi
4,2025-03-01 00:01:44,1.50,8.6,17.20,8.266667,0,5,3,Taxi
...,...,...,...,...,...,...,...,...,...
3228589,2025-03-31 23:08:46,1.38,10.0,18.75,7.933333,23,0,3,Taxi
3228590,2025-03-31 23:12:36,4.42,22.6,34.02,19.433333,23,0,3,Taxi
3228591,2025-03-31 23:39:28,0.30,5.1,13.00,3.183333,23,0,3,Taxi
3228592,2025-03-31 23:49:49,6.20,27.5,33.25,16.933333,23,0,3,Taxi


In [66]:
# Łączymy taxi z Uberem i eksportujemy do csv
#     żeby było wygodniej wygodniej do statystyki opisowej
uber = pd.read_csv("uber_do_statOp.csv")

# Robimy wspólne kolumny
uber_common = uber.rename(columns={
    "pickup_datetime": "datetime",
    "trip_miles": "distance",
    "base_passenger_fare": "base_fare",
    "hvfhs_license_num": "provider",
})

uber_common["provider"] = "Uber"

taxi_common = taxi.rename(columns={
    "tpep_pickup_datetime": "datetime",
    "trip_distance": "distance",
    "fare_amount": "base_fare",
    "type": "provider",
})

# Zostawiamy tylko wspolne kolumny
common_cols = [
    "provider", "datetime", "distance",
    "trip_time_mins", "base_fare", 
    "total_amount", "hour", "weekday", "month"
]

uber_common = uber_common[common_cols]
taxi_common = taxi_common[common_cols]

# Łączymy w jeden zbior
wszystkie = pd.concat([uber_common, taxi_common], ignore_index=True)

In [67]:
wszystkie

,provider,datetime,distance,trip_time_mins,base_fare,total_amount,hour,weekday,month
0,Uber,2025-03-01 00:17:10,1.47,9.983333,23.83,30.75,0,5,3
1,Uber,2025-03-01 00:33:19,3.23,12.000000,18.05,24.33,0,5,3
2,Uber,2025-03-01 00:35:45,4.68,22.183333,41.67,50.91,0,5,3
3,Uber,2025-03-01 00:45:35,2.90,15.650000,59.79,70.79,0,5,3
4,Uber,2025-03-01 00:26:12,3.04,18.066667,21.92,24.58,0,5,3
...,...,...,...,...,...,...,...,...,...
6064388,Taxi,2025-03-31 23:08:46,1.38,7.933333,10.00,18.75,23,0,3
6064389,Taxi,2025-03-31 23:12:36,4.42,19.433333,22.60,34.02,23,0,3
6064390,Taxi,2025-03-31 23:39:28,0.30,3.183333,5.10,13.00,23,0,3
6064391,Taxi,2025-03-31 23:49:49,6.20,16.933333,27.50,33.25,23,0,3


In [68]:
# Eksportujemy do csv
wszystkie.to_csv("wszystkie_do_statOp.csv", index=False)